In [81]:
from __future__ import annotations

import sys
from datetime import datetime
from typing import Optional

import pandas as pd
from cassandra.cluster import Cluster, NoHostAvailable  # pyasyncore 설치로 Python 3.12 호환
from cassandra.policies import DCAwareRoundRobinPolicy
from cassandra.query import SimpleStatement

In [82]:
def load_from_cassandra(
    host: str = "localhost",
    port: int = 9042,
    keyspace: str = "dlim",
    table: str = "ais_class_a_dynamic",
    mmsi_list: Optional[list[str]] = None,
    start_dt: Optional[datetime] = None,
    end_dt: Optional[datetime] = None,
    limit: int = 100_000,
) -> pd.DataFrame:
    """
    Cassandra에서 AIS 데이터를 조회해 pandas DataFrame으로 반환한다.

    Parameters
    ----------
    host       : Cassandra 호스트
    port       : Cassandra 포트
    keyspace   : 키스페이스 이름
    table      : 테이블 이름
    mmsi_list  : 조회할 MMSI 목록 (None이면 전체)
    start_dt   : date_bucket 시작 시각 (포함)
    end_dt     : date_bucket 종료 시각 (포함)
    limit      : 최대 레코드 수

    Returns
    -------
    pd.DataFrame : 조회된 레코드
    """
    print(f"[Cassandra] 연결: {host}:{port} / keyspace={keyspace}")

    try:
        cluster = Cluster(
            contact_points=[host],
            port=port,
            load_balancing_policy=DCAwareRoundRobinPolicy(),
            protocol_version=4,
        )
        session = cluster.connect(keyspace)
    except NoHostAvailable as exc:
        print(f"[오류] Cassandra 연결 실패: {exc}")
        sys.exit(1)
    except Exception as exc:
        print(f"[오류] 예기치 않은 연결 오류: {exc}")
        sys.exit(1)

    COLUMNS = (
        "mmsi, date_bucket, received_at, msg_type, sog, cog, "
        "nav_status, longitude, latitude, rssi, slot_num, snr"
    )

    rows: list[dict] = []

    try:
        if mmsi_list:
            # MMSI 별로 파티션 키 범위 쿼리 → ALLOW FILTERING 불필요
            for mmsi in mmsi_list:
                cql, params = _build_single_mmsi_query(
                    table, COLUMNS, mmsi, start_dt, end_dt, limit
                )
                stmt = SimpleStatement(cql, fetch_size=1000)
                result = session.execute(stmt, params)
                rows.extend(_rows_to_dicts(result))
                if len(rows) >= limit:
                    rows = rows[:limit]
                    break
        else:
            # 전체 조회: date_bucket 범위만 적용 → ALLOW FILTERING 불가피
            cql, params = _build_full_scan_query(
                table, COLUMNS, start_dt, end_dt, limit
            )
            stmt = SimpleStatement(cql, fetch_size=1000)
            result = session.execute(stmt, params)
            rows = _rows_to_dicts(result)
    finally:
        cluster.shutdown()

    if not rows:
        print("[조회] 결과 없음 — 빈 DataFrame 반환")
        return pd.DataFrame(columns=COLUMNS.replace(" ", "").split(","))

    df = pd.DataFrame(rows)

    # datetime 변환 (Cassandra driver는 대부분 datetime으로 반환하나 명시적 보장)
    for col in ("date_bucket", "received_at"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")

    print(f"[조회] {table} → {len(df):,}건 로드 완료")
    return df


def _build_single_mmsi_query(
    table: str,
    columns: str,
    mmsi: str,
    start_dt: Optional[datetime],
    end_dt: Optional[datetime],
    limit: int,
) -> tuple[str, list]:
    """단일 MMSI에 대한 파티션 키 쿼리를 생성한다."""
    cql = f"SELECT {columns} FROM {table} WHERE mmsi = %s"
    params: list = [mmsi]

    if start_dt is not None:
        cql += " AND date_bucket >= %s"
        params.append(start_dt)
    if end_dt is not None:
        cql += " AND date_bucket <= %s"
        params.append(end_dt)

    cql += f" LIMIT {limit}"
    return cql, params


def _build_full_scan_query(
    table: str,
    columns: str,
    start_dt: Optional[datetime],
    end_dt: Optional[datetime],
    limit: int,
) -> tuple[str, list]:
    """전체 스캔 쿼리를 생성한다. 날짜 범위가 있으면 ALLOW FILTERING을 추가한다."""
    cql = f"SELECT {columns} FROM {table}"
    params: list = []
    conditions: list[str] = []

    if start_dt is not None:
        conditions.append("date_bucket >= %s")
        params.append(start_dt)
    if end_dt is not None:
        conditions.append("date_bucket <= %s")
        params.append(end_dt)

    if conditions:
        cql += " WHERE " + " AND ".join(conditions) + " ALLOW FILTERING"

    cql += f" LIMIT {limit}"
    return cql, params

def _rows_to_dicts(result) -> list[dict]:
    """cassandra ResultSet을 dict 리스트로 변환한다."""
    return [
        {
            "mmsi":        row.mmsi,
            "date_bucket": row.date_bucket,
            "received_at": row.received_at,
            "msg_type":    row.msg_type,
            "sog":         row.sog,
            "cog":         row.cog,
            "nav_status":  row.nav_status,
            "longitude":   row.longitude,
            "latitude":    row.latitude,
            "rssi":        row.rssi,
            "slot_num":    row.slot_num,
            "snr":         row.snr,
        }
        for row in result
    ]

def split_by_mmsi(df: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    전체 DataFrame을 MMSI 단위로 분리한다.

    Parameters
    ----------
    df : load_from_cassandra()가 반환한 DataFrame

    Returns
    -------
    dict[str, pd.DataFrame]
        key   : mmsi 문자열
        value : 해당 선박의 레코드 (date_bucket 오름차순 정렬)
    """
    if df.empty:
        return {}

    mmsi_dfs: dict[str, pd.DataFrame] = {}
    for mmsi, group in df.groupby("mmsi", sort=False):
        mmsi_dfs[str(mmsi)] = (
            group[["date_bucket", "received_at", "msg_type", "sog", "cog", "nav_status", "longitude", "latitude", "rssi", "slot_num", "snr"]]
            .sort_values("date_bucket")
            .reset_index(drop=True)
        )

    return mmsi_dfs

def print_summary(mmsi_dfs: dict[str, pd.DataFrame]) -> None:
    """
    선박별 레코드 수, 관측 시간 등 요약 정보를 콘솔에 출력한다.
    """
    if not mmsi_dfs:
        print("[요약] 데이터 없음")
        return

    total_ships = len(mmsi_dfs)
    total_records = sum(len(v) for v in mmsi_dfs.values())

    print(f"\n[요약] 총 {total_ships:,}척 / {total_records:,}건")

    header = (
        f"  {'MMSI':<14} {'count':>6}   "
        f"{'start_time':<22} {'end_time':<22} {'duration(min)':>13}"
    )
    print(header)
    print("  " + "-" * (len(header) - 2))

    # 레코드 수 내림차순 정렬
    for mmsi, sub_df in sorted(
        mmsi_dfs.items(), key=lambda x: len(x[1]), reverse=True
    ):
        count = len(sub_df)
        start_time = sub_df["date_bucket"].iloc[0]
        end_time = sub_df["date_bucket"].iloc[-1]

        duration_min = (
            (end_time - start_time).total_seconds() / 60.0
            if pd.notna(start_time) and pd.notna(end_time)
            else float("nan")
        )

        # timezone-aware datetime을 보기 좋게 포맷
        fmt = "%Y-%m-%d %H:%M:%S"
        start_str = start_time.strftime(fmt) if pd.notna(start_time) else "N/A"
        end_str = end_time.strftime(fmt) if pd.notna(end_time) else "N/A"

        print(
            f"  {mmsi:<14} {count:>6}   "
            f"{start_str:<22} {end_str:<22} {duration_min:>13.1f}"
        )

In [83]:
# ── 조회 파라미터 ──────────────────────────────────────────────────────────
MMSI_LIST = None          # 특정 선박만: ["538007769", "440100123"]
START_DT  = None          # datetime(2026, 3, 20, 12, 0, 0)
END_DT    = None          # datetime(2026, 3, 20, 13, 0, 0)
LIMIT     = 100_000

# ── 데이터 로드 ────────────────────────────────────────────────────────────
df = load_from_cassandra(
    mmsi_list=MMSI_LIST,
    start_dt=START_DT,
    end_dt=END_DT,
    limit=LIMIT,
)

# ── 선박별 분리 ────────────────────────────────────────────────────────────
mmsi_dfs = split_by_mmsi(df)
print_summary(mmsi_dfs)
print(df)

[Cassandra] 연결: localhost:9042 / keyspace=dlim
[조회] ais_class_a_dynamic → 58,167건 로드 완료

[요약] 총 157척 / 58,167건
  MMSI            count   start_time             end_time               duration(min)
  -----------------------------------------------------------------------------------
  440016000        1803   2026-03-20 11:59:24    2026-03-20 12:59:28             60.1
  636021910        1802   2026-03-20 11:59:24    2026-03-20 12:59:28             60.1
  256459000        1795   2026-03-20 11:59:23    2026-03-20 12:59:29             60.1
  413703960        1783   2026-03-20 11:59:23    2026-03-20 12:59:29             60.1
  538007769        1773   2026-03-20 11:59:23    2026-03-20 12:59:29             60.1
  636015991        1747   2026-03-20 11:59:23    2026-03-20 12:59:29             60.1
  668116244        1056   2026-03-23 14:59:25    2026-03-23 15:34:35             35.2
  273428130        1053   2026-03-23 14:59:26    2026-03-23 15:34:36             35.2
  636016053        1051   202

In [84]:
df["mmsi"].sort_values().drop_duplicates()

49186    111131433
19821    128864111
4710     140314169
51411    156106498
32227    157419116
           ...    
24931    855215235
11346    903811553
47347    914070447
21871    964638703
34666    988864582
Name: mmsi, Length: 157, dtype: str

In [92]:
import datetime

df_mar23 = df[df["date_bucket"].dt.date == datetime.date(2026, 3, 25)].copy()

#df_sort = df.sort_values(["mmsi","date_bucket"])

df_sort = df_mar23.sort_values(["date_bucket"])

In [93]:
with pd.option_context("display.max_rows", None):
    display(df_sort[["mmsi","date_bucket","slot_num"]])

,mmsi,date_bucket,slot_num
16768,220059000,2026-03-25 11:02:02.758000+00:00,114
52883,563141200,2026-03-25 11:02:02.866000+00:00,113
42885,636020892,2026-03-25 11:02:03.853000+00:00,160
31343,356758000,2026-03-25 11:02:04.777000+00:00,193
7485,352001515,2026-03-25 11:02:04.936000+00:00,199
10490,477947700,2026-03-25 11:02:05.026000+00:00,213
22663,311054300,2026-03-25 11:02:05.142000+00:00,201
22520,636020892,2026-03-25 11:02:05.865000+00:00,228
32116,636020590,2026-03-25 11:02:05.971000+00:00,233
33921,538005269,2026-03-25 11:02:06.343000+00:00,240


In [94]:
with pd.option_context("display.max_rows", None):
    display(df_sort[df_sort["mmsi"] == "414061000"])

,mmsi,date_bucket,received_at,msg_type,sog,cog,nav_status,longitude,latitude,rssi,slot_num,snr


In [95]:
def get_NI(sog_raw):
    """SOG는 ×10 저장이므로 /10 후 NI 반환"""
    sog = sog_raw / 10.0
    if sog == 0:
        return None      # 정박/계류 → 별도 확인
    elif sog < 14:
        return 375       # 보고주기 10s → 2250/6
    elif sog < 23:
        return 225       # 보고주기  6s → 2250/10
    else:
        return 75        # 보고주기  2s → 2250/30

df_check = df_sort.copy()

# 선박별 이전 행의 slot_num, sog
df_check["prev_slot"] = df_check.groupby("mmsi")["slot_num"].shift(1)
df_check["prev_sog"]  = df_check.groupby("mmsi")["sog"].shift(1)

# 첫 번째 행(NaN) 제거
df_check = df_check.dropna(subset=["prev_slot", "prev_sog"]).copy()
df_check["prev_slot"] = df_check["prev_slot"].astype(int)

# NI 계산
df_check["NI"] = df_check["prev_sog"].apply(get_NI)

In [96]:
def slot_diff_wrap(prev, curr, max_slot=2250):
    diff = int(curr) - int(prev)
    if diff < 0:
        diff += max_slot   # 0으로 초기화된 경우 보정
    return diff

df_check["slot_diff"] = df_check.apply(
    lambda r: slot_diff_wrap(r["prev_slot"], r["slot_num"]), axis=1
)
df_check

,mmsi,date_bucket,received_at,msg_type,sog,cog,nav_status,longitude,latitude,rssi,slot_num,snr,prev_slot,prev_sog,NI,slot_diff
22520,636020892,2026-03-25 11:02:05.865000+00:00,2026-03-25 11:42:53.378000+00:00,3,264.0,301.0,15.0,129.41673,34.92316,-109.000000,228,11.400000,160,264.0,75,68
27626,356758000,2026-03-25 11:02:06.777000+00:00,2026-03-25 11:42:53.519000+00:00,3,297.0,990.0,3.0,128.74689,35.05979,-112.800003,261,6.400000,193,297.0,75,68
19793,636020892,2026-03-25 11:02:07.852000+00:00,2026-03-25 11:42:53.582000+00:00,3,264.0,301.0,3.0,129.41673,34.92316,-114.599998,303,4.600000,228,264.0,75,75
5007,440310510,2026-03-25 11:02:08.370000+00:00,2026-03-25 11:42:53.598000+00:00,3,259.0,2969.0,4.0,129.24993,35.09598,-100.699997,325,19.500000,257,259.0,75,68
56378,440000340,2026-03-25 11:02:08.523000+00:00,2026-03-25 11:42:53.613000+00:00,3,272.0,660.0,9.0,129.22618,34.94833,-103.300003,323,16.299999,255,272.0,75,68
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12150,538006103,2026-03-25 11:21:38.938000+00:00,2026-03-25 11:45:25.485000+00:00,1,186.0,1914.0,15.0,129.18676,35.12498,-102.400002,1456,16.500000,1216,186.0,225,240
7352,312716000,2026-03-25 11:21:39.147000+00:00,2026-03-25 11:45:25.501000+00:00,1,234.0,2587.0,10.0,128.78993,35.06431,-105.400002,1463,14.900000,1389,234.0,75,74
29000,352004507,2026-03-25 11:21:39.174000+00:00,2026-03-25 11:45:25.518000+00:00,1,205.0,2472.0,12.0,129.35102,35.28782,-113.900002,1454,5.100000,1230,205.0,225,224
4355,416044000,2026-03-25 11:21:39.461000+00:00,2026-03-25 11:45:25.533000+00:00,1,266.0,1829.0,13.0,128.98792,34.89233,-108.599998,1474,10.900000,1399,266.0,75,75


In [97]:
def is_slot_valid(row):
    NI = row["NI"]
    if NI is None:
        return None   # SOG=0은 별도 검토
    margin = 0.2 * NI
    diff   = row["slot_diff"]
    # slot_diff가 [0.8*NI, 1.2*NI] 범위 안에 있어야 정상
    return (NI - margin) <= diff <= (NI + margin)

df_check["slot_valid"] = df_check.apply(is_slot_valid, axis=1)
df_check

,mmsi,date_bucket,received_at,msg_type,sog,cog,nav_status,longitude,latitude,rssi,slot_num,snr,prev_slot,prev_sog,NI,slot_diff,slot_valid
22520,636020892,2026-03-25 11:02:05.865000+00:00,2026-03-25 11:42:53.378000+00:00,3,264.0,301.0,15.0,129.41673,34.92316,-109.000000,228,11.400000,160,264.0,75,68,True
27626,356758000,2026-03-25 11:02:06.777000+00:00,2026-03-25 11:42:53.519000+00:00,3,297.0,990.0,3.0,128.74689,35.05979,-112.800003,261,6.400000,193,297.0,75,68,True
19793,636020892,2026-03-25 11:02:07.852000+00:00,2026-03-25 11:42:53.582000+00:00,3,264.0,301.0,3.0,129.41673,34.92316,-114.599998,303,4.600000,228,264.0,75,75,True
5007,440310510,2026-03-25 11:02:08.370000+00:00,2026-03-25 11:42:53.598000+00:00,3,259.0,2969.0,4.0,129.24993,35.09598,-100.699997,325,19.500000,257,259.0,75,68,True
56378,440000340,2026-03-25 11:02:08.523000+00:00,2026-03-25 11:42:53.613000+00:00,3,272.0,660.0,9.0,129.22618,34.94833,-103.300003,323,16.299999,255,272.0,75,68,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12150,538006103,2026-03-25 11:21:38.938000+00:00,2026-03-25 11:45:25.485000+00:00,1,186.0,1914.0,15.0,129.18676,35.12498,-102.400002,1456,16.500000,1216,186.0,225,240,True
7352,312716000,2026-03-25 11:21:39.147000+00:00,2026-03-25 11:45:25.501000+00:00,1,234.0,2587.0,10.0,128.78993,35.06431,-105.400002,1463,14.900000,1389,234.0,75,74,True
29000,352004507,2026-03-25 11:21:39.174000+00:00,2026-03-25 11:45:25.518000+00:00,1,205.0,2472.0,12.0,129.35102,35.28782,-113.900002,1454,5.100000,1230,205.0,225,224,True
4355,416044000,2026-03-25 11:21:39.461000+00:00,2026-03-25 11:45:25.533000+00:00,1,266.0,1829.0,13.0,128.98792,34.89233,-108.599998,1474,10.900000,1399,266.0,75,75,True


In [98]:
print("=== 전체 슬롯 검증 결과 ===")
print(df_check["slot_valid"].value_counts(dropna=False))

# 선박별 이상 비율
summary = df_check.groupby("mmsi")["slot_valid"].agg(
    total="count",
    valid=lambda x: x.sum(),
    invalid=lambda x: (x == False).sum(),
    sog_zero=lambda x: x.isna().sum(),
)
summary["invalid_rate(%)"] = (summary["invalid"] / summary["total"] * 100).round(2)
print("\n=== 선박별 요약 ===")
print(summary)

# 이상 행만 따로 확인
df_invalid = df_check[df_check["slot_valid"] == False]
print(f"\n이상 레코드 수: {len(df_invalid)}")
print(df_invalid[["mmsi", "date_bucket", "prev_slot", "slot_num", "slot_diff", "NI", "prev_sog"]])

=== 전체 슬롯 검증 결과 ===
slot_valid
True     9541
False      28
Name: count, dtype: int64

=== 선박별 요약 ===
           total  valid  invalid  sog_zero  invalid_rate(%)
mmsi                                                       
219357000    192    192        0         0             0.00
219710000    574    573        1         0             0.17
273415260    191    191        0         0             0.00
311054300    195    195        0         0             0.00
312716000    585    584        1         0             0.17
352002591    195    195        0         0             0.00
352004507    195    195        0         0             0.00
352927000    583    581        2         0             0.34
353282000    191    191        0         0             0.00
356758000    586    585        1         0             0.17
416044000    576    576        0         0             0.00
440000340    583    580        3         0             0.51
440021790    115    115        0         0             0.00

In [99]:
summary

,total,valid,invalid,sog_zero,invalid_rate(%)
mmsi,,,,,
219357000,192,192,0,0,0.00
219710000,574,573,1,0,0.17
273415260,191,191,0,0,0.00
311054300,195,195,0,0,0.00
312716000,585,584,1,0,0.17
352002591,195,195,0,0,0.00
352004507,195,195,0,0,0.00
352927000,583,581,2,0,0.34
353282000,191,191,0,0,0.00


In [100]:
with pd.option_context("display.max_rows", None):
    display(df_check[(df_check["mmsi"] == "414061000")&(df_check["slot_valid"] == False)][["date_bucket","slot_num","slot_diff","slot_valid"]])


,date_bucket,slot_num,slot_diff,slot_valid


In [101]:
cols = ["date_bucket", "slot_num", "slot_diff", "slot_valid"]

# 해당 선박만 추출 후 위치 인덱스 리셋
df_ship = df_check[df_check["mmsi"] == "273841100"].reset_index(drop=True)

# False인 행의 위치(position) 추출
false_pos = df_ship[df_ship["slot_valid"] == False].index.tolist()

# 앞뒤 n행씩 위치 확장
n = 2
expanded_pos = set()
for pos in false_pos:
    for i in range(max(0, pos - n), min(len(df_ship), pos + n + 1)):
        expanded_pos.add(i)

expanded_pos = sorted(expanded_pos)

# 강조 출력
def highlight_false(row):
    if row["slot_valid"] == False:
        return ["background-color: #ffcccc"] * len(row)
    return [""] * len(row)

with pd.option_context("display.max_rows", None):
    display(
        df_ship.loc[expanded_pos, cols]
        .style.apply(highlight_false, axis=1)
    )

,date_bucket,slot_num,slot_diff,slot_valid


In [102]:
cols = ["date_bucket", "slot_num", "slot_diff", "slot_valid"]
def highlight_false(row):
  if row["slot_valid"] == False:
      return ["background-color: #ffcccc"] * len(row)
  return [""] * len(row)
n = 2  # 앞뒤 행 수
for mmsi in sorted(df_check["mmsi"].unique()):
  df_ship = df_check[df_check["mmsi"] == mmsi].reset_index(drop=True)
  # False인 행이 없으면 스킵
  false_pos = df_ship[df_ship["slot_valid"] == False].index.tolist()
  if not false_pos:
      continue
  print(f"\n{'='*60}")
  print(f"MMSI: {mmsi}  |  이상 건수: {len(false_pos)}건")
  print(f"{'='*60}")
  # 앞뒤 n행씩 확장
  expanded_pos = set()
  for pos in false_pos:
      for i in range(max(0, pos - n), min(len(df_ship), pos + n + 1)):
          expanded_pos.add(i)
  expanded_pos = sorted(expanded_pos)
  with pd.option_context("display.max_rows", None):
      display(
          df_ship.loc[expanded_pos, cols]
          .style.apply(highlight_false, axis=1)
      )


MMSI: 219710000  |  이상 건수: 1건


,date_bucket,slot_num,slot_diff,slot_valid
267,2026-03-25 11:11:25.975000+00:00,987,75,True
268,2026-03-25 11:11:27.968000+00:00,1062,75,True
269,2026-03-25 11:11:29.978000+00:00,1120,58,False
270,2026-03-25 11:11:31.972000+00:00,1193,73,True
271,2026-03-25 11:11:33.964000+00:00,1268,75,True



MMSI: 312716000  |  이상 건수: 1건


,date_bucket,slot_num,slot_diff,slot_valid
272,2026-03-25 11:11:13.153000+00:00,487,76,True
273,2026-03-25 11:11:15.155000+00:00,561,74,True
274,2026-03-25 11:11:19.161000+00:00,712,151,False
275,2026-03-25 11:11:21.156000+00:00,788,76,True
276,2026-03-25 11:11:23.155000+00:00,862,74,True



MMSI: 352927000  |  이상 건수: 2건


...

,date_bucket,slot_num,slot_diff,slot_valid
366,2026-03-25 11:14:20.527000+00:00,765,76,True
367,2026-03-25 11:14:22.526000+00:00,838,73,True
368,2026-03-25 11:14:26.530000+00:00,988,150,False
369,2026-03-25 11:14:28.531000+00:00,1064,76,True
370,2026-03-25 11:14:30.523000+00:00,1139,75,True
533,2026-03-25 11:19:56.536000+00:00,2113,73,True
534,2026-03-25 11:19:58.532000+00:00,2188,75,True
535,2026-03-25 11:20:02.531000+00:00,89,151,False
536,2026-03-25 11:20:04.527000+00:00,163,74,True
537,2026-03-25 11:20:06.530000+00:00,238,75,True



MMSI: 636020892  |  이상 건수: 2건


,date_bucket,slot_num,slot_diff,slot_valid
87,2026-03-25 11:04:59.865000+00:00,3,75,True
88,2026-03-25 11:05:01.857000+00:00,78,75,True
89,2026-03-25 11:05:03.855000+00:00,137,59,False
90,2026-03-25 11:05:05.853000+00:00,217,80,True
91,2026-03-25 11:05:07.867000+00:00,287,70,True
463,2026-03-25 11:17:31.855000+00:00,1189,75,True
464,2026-03-25 11:17:33.853000+00:00,1262,73,True
465,2026-03-25 11:17:37.864000+00:00,1423,161,False
466,2026-03-25 11:17:39.857000+00:00,1487,64,True
467,2026-03-25 11:17:41.853000+00:00,1564,77,True



MMSI: 636022201  |  이상 건수: 18건


,date_bucket,slot_num,slot_diff,slot_valid
267,2026-03-25 11:11:26.581000+00:00,1015,75,True
268,2026-03-25 11:11:28.579000+00:00,1090,75,True
269,2026-03-25 11:11:30.583000+00:00,1141,51,False
270,2026-03-25 11:11:36.577000+00:00,1373,232,False
271,2026-03-25 11:11:38.588000+00:00,1448,75,True
272,2026-03-25 11:11:40.584000+00:00,1520,72,True
281,2026-03-25 11:11:58.585000+00:00,2190,63,True
282,2026-03-25 11:12:00.577000+00:00,18,78,True
283,2026-03-25 11:12:06.578000+00:00,248,230,False
284,2026-03-25 11:12:08.578000+00:00,327,79,True
